# Challenge Overview

Build machine learning (ML) models to detect misinformation in Portuguese news articles, and
leverage interpretability and explainability methods to analyze results on the FakeNews-PT
dataset.

# 1 Model Training & Evaluation (10 points)

a) Extract TF-IDF features from the text with a maximum number of features (terms) set to 5000.
Make sure to add smoothing for out-of-vocabulary (OOV) words (idf smoothing). Define the
minimum and maximum number of documents a term must appear in as min_df=10, and the
maximum proportion of documents a term can appear in as max_df=0.9.

In [103]:
# Feature selection through Tfidfvectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Metrics and Kfold
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import RocCurveDisplay, auc

# Required models
from sklearn.tree import DecisionTreeClassifier 
from sklearn import tree
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans

# Data manipulation
import pandas as pd

# Visualization
import matplotlib as plt

# Numerical Computing
import numpy as np

# import spacy

import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /home/peter-
[nltk_data]     parker/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [78]:
from nltk.corpus import stopwords
PORTUGUESE_STOPWORDS = stopwords.words('portuguese')

RANDOM_STATE = 42

In [79]:
# Obtain all the text documents as a pandas data frame (train, validation and test)
train_fake_news_df = pd.read_csv("../data/train.csv")
val_fake_news_df = pd.read_csv("../data/val.csv")
test_fake_news_df = pd.read_csv("../data/test.csv")

# Separate text documents from their labels (train, validation and test)
train_news_texts = train_fake_news_df['Text']
train_news_labels = train_fake_news_df['Label']

val_news_texts = train_fake_news_df['Text']
val_news_labels = train_fake_news_df['Label']

test_news_texts = train_fake_news_df['Text']
test_news_labels = train_fake_news_df['Label']

In [80]:
print("News Texts:\n")
print(train_news_texts)
print("\nNews Labels:\n")
print(train_news_labels)


News Texts:

0        PJ em operação internacional de tráfico de dro...
1        O texto foi criado - numa universidade da Repú...
2        Nunca pague multas por conduzir sem carta ou s...
3        Depois de António Guterres, uma estátua de D. ...
4        O MAR, uma droga gratuita que cura pelo menos ...
                               ...                        
50582    FMI recomenda reforma para evitar "acumulação ...
50583    Governo diz que o estado vai indemnizar as vít...
50584    Homem com cancro terminal realiza último desej...
50585    Grupo Impresa tem dívida de 189,1 milhões de e...
50586    Mãe fica chocada ao perceber quem estava a seg...
Name: Text, Length: 50587, dtype: object

News Labels:

0        1
1        1
2        0
3        0
4        0
        ..
50582    1
50583    0
50584    0
50585    0
50586    0
Name: Label, Length: 50587, dtype: int64


In [81]:
# Lemanize texts
# nlp = spacy.load("pt_core_news_lg")
#def lemmatize_text(text):
    #doc = nlp(text)
    #return " ".join([token.lemma_ for token in doc if not token.is_punct and not token.is_space])
#lemmatized_texts = [lemmatize_text(t) for t in texts]

#print(lemmatized_texts)

In [82]:
# Characteristics of the tf_idf feature selection that were requested
MAX_FEATURES = 5000
MIN_DF = 10
MAX_DF = 0.9
SMOOTH_IDF = True
STOP_WORDS = PORTUGUESE_STOPWORDS

# Optional characteristics that improve feature selection
LOWER_CASE = True


# Tf-idf vectorizer initialization
tfidf_vectorizer = TfidfVectorizer(
    max_features= MAX_FEATURES,
    min_df= MIN_DF,
    max_df= MAX_DF,
    smooth_idf= SMOOTH_IDF,
    stop_words= STOP_WORDS,
    lowercase= LOWER_CASE
)

In [83]:
# Obtain the matrix of df and idf frequencies for training
train_tfidf = tfidf_vectorizer.fit_transform(train_news_texts)

# Get the name of the features that were determined
feature_names = tfidf_vectorizer.get_feature_names_out()

print("TF-IDF matrix shape:", train_tfidf.shape)
print("Exemplo de termos:", feature_names[:20])
train_tfidf

TF-IDF matrix shape: (50587, 5000)
Exemplo de termos: ['00' '000' '01' '02' '03' '04' '05' '06' '07' '08' '09' '10' '100' '11'
 '110' '116' '12' '120' '125' '13']


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5096069 stored elements and shape (50587, 5000)>

In [84]:
# Transform the validation and testing data
val_tfidf = tfidf_vectorizer.transform(val_news_texts)
test_tfidf = tfidf_vectorizer.transform(test_news_texts)

b) Train the following models using 5-fold cross-validation, tune key hyperparameters systematically
(e.g., regularization strength λ, tree depth), and document your hyperparameter search process.
1) Decision Tree
2) Gaussian Naive Bayes
3) Logistic Regression with L2 regularization
4) Logistic Regression with L1 regularization
5) Multi-Layer Perceptron (MLP)

In [101]:
# KFold that will be used for All the models
K_FOLDS = 5
SHUFFLE = True

skf = StratifiedKFold(n_splits= K_FOLDS, shuffle= SHUFFLE, random_state= RANDOM_STATE)

scoring = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score, average='weighted'),
    'recall': make_scorer(recall_score, average='weighted'),
    'f1': make_scorer(f1_score, average='weighted')
}

In [86]:
# Decision Tree
#depths_allowed = [2, 3, 4, 5, 10, 25, 50, 100]
#min_samples_split_allowed = range(2, 101)

#best_valid_acc = -1.0
#best_param = None
#best_model = None

# Go thorugh various tree depths
#for depth in depths_allowed:

    # Go thorugh various min samples split values
 #   for split in min_samples_split_allowed:
  #      decision_tree_model = DecisionTreeClassifier(min_samples_split=split, max_depth= depth, random_state=1)
   #     decision_tree_model.fit(train_tfidf, train_news_labels)
    #    val_labels_pred = decision_tree_model.predict(val_tfidf)
     #   valid_acc = accuracy_score(val_news_labels, val_labels_pred)

        # Obtain the best Decision classifier based on accuracy
      #  if valid_acc > best_valid_acc:
       #     best_valid_acc = valid_acc
        #    best_param = [depth, split]
         #   best_model = decision_tree_model

# Test with the training data
#test_labels_pred = best_model.predict(test_tfidf)
#test_acc = accuracy_score(test_news_labels, test_labels_pred)
#test_acc

In [87]:
#figure = plt.figure(figsize=(12, 6))
#tree.plot_tree(best_model, feature_names=tfidf_vectorizer.get_feature_names_out(), class_names=[str(c) for c in decision_tree_model.classes_], impurity=False)
#plt.show()

In [ ]:
# Gaussian Naive Bayes
gaussian_NB = GaussianNB()

# Perform 5-fold CV
cv_results = cross_validate(
    gaussian_NB,
    train_tfidf.toarray(),  # GaussianNB requires dense matrix
    train_news_labels,
    cv=skf,
    scoring=scoring,
    return_train_score=False
)

results_df = pd.DataFrame({
    'Fold': np.arange(1, 6),
    'Accuracy': cv_results['test_accuracy'],
    'Precision': cv_results['test_precision'],
    'Recall': cv_results['test_recall'],
    'F1': cv_results['test_f1']
})

results_df


In [89]:
# Logistic Regression with L2 regularization

In [90]:
# Logistic Regression with L1 regularization

In [91]:
# Multi-layer Perceptron (MLP)

c) Create a comparison table with test metrics: Accuracy, Precision, Recall, and F1-score. For the
best classifier, draw its ROC curve and compute AUC.

# 2 Model Interpretation (7 points)

a) For your best Logistic Regression model, extract and visualize the weights in a bar plot:
1) Top 10 words most indicative of fake news
2) Top 10 words most indicative of real news

b) Compare L1 vs L2 regularized models: How many features have non-zero weights in each? What
does this tell you about feature selection? When would you prefer L1 vs L2 regularization for
text classification?

c) For your best Logistic Regression model, select samples in the validation set with ID 2921,
2437, 5557, 1697, and extract explanations with LIME (Ribeiro et al., 2016; Lundberg and
Lee, 2017).

d) For your MLP, select samples in the validation set with ID 2921, 2437, 5557, 1697, and
extract explanations with LIME and permutation importance. For permutation importance,
select 1K random samples. Visualize the results and discuss their differences.

# 3 Clustering (3 points)

a) Apply K-Means with K=5 on your training set.

b) Inspect 3 documents closest to each centroid, and afterwards, assign semantic labels to each
cluster (e.g., “political fake news”, “health misinformation”).

c) Visualize clusters in 2D using PCA. Create two plots: one colored by cluster assignment, one by
true label.